In [ ]:
# Run this if required libraries are not already installed
!pip install numpy xarray matplotlib cartopy seaborn netCDF4


In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import numpy as np
import seaborn as sns
from netCDF4 import Dataset
import cartopy.feature as cfeature


In [ ]:
# NOAA ERSST v5 dataset — Extended Reconstructed Sea Surface Temperature
# Data Source: NOAA PSL (https://psl.noaa.gov/)
# We read directly from NOAA's THREDDS server

baseURL = 'http://www.esrl.noaa.gov'
catalogURL = '/psd/thredds/dodsC/Datasets/noaa.ersst.v5/sst.mnmean.nc'
dataset_url = baseURL + catalogURL

# Load the dataset
nc = Dataset(dataset_url)
sstID = xr.open_dataset(xr.backends.NetCDF4DataStore(nc))


In [ ]:
# Get the 'sst' variable from the dataset
sst = sstID['sst']

# Find the most recent time index
mostRecent = len(sst.time.values) - 1

# Get sea surface temperature for that time step
recentSST = sst.isel(time=mostRecent)


In [ ]:
# Set contour levels (°C)
sstmin = 0
sstmax = 30
levels = np.linspace(sstmin, sstmax, 21)

# Use seaborn’s icefire colormap
cmap = sns.color_palette("icefire", as_cmap=True)

# Plot SST data on an Orthographic map projection (centered on Asia)
fig = plt.figure(figsize=[12, 6], facecolor='none')
ax = plt.subplot(1, 1, 1, projection=ccrs.Orthographic(central_longitude=90, central_latitude=0), facecolor='none')

# Plot filled contours of SST data
contour = recentSST.plot.contourf(
    levels=levels,
    cmap=cmap,
    transform=ccrs.PlateCarree(),
    ax=ax,
    add_colorbar=False
)

# Add coastlines and country borders
ax.coastlines('10m')
ax.add_feature(cfeature.BORDERS, edgecolor='black')

# Optional: Fill countries with light grey
countries = cfeature.NaturalEarthFeature(
    category='cultural',
    name='admin_0_countries',
    scale='10m',
    facecolor='lightgrey'
)
ax.add_feature(countries, edgecolor='black')

# Optional: Add colorbar
cbar = plt.colorbar(contour, ax=ax, orientation='horizontal', pad=0.05)
cbar.set_label('Sea Surface Temperature (°C)', fontsize=12)

# Optional: Add title
plt.title('Global Sea Surface Temperature (Most Recent Month)', fontsize=14, weight='bold')

# Save the figure if needed
plt.savefig('SST_plot_asia.png', dpi=300, bbox_inches='tight', transparent=True)

# Show the plot
plt.show()
